# 가전제품 PDF 매뉴얼 청킹 + RDB/벡터DB 하이브리드 질의응답

이 노트북은 다음 과정을 실제로 실행한 결과를 정리한다:

1. PDF가 OCR이 필요한 스캔본인지 진단
2. 청킹 방식 비교: 폰트 크기 추정 vs PDF 내장 목차(TOC) 기반
3. 청크를 `{id, text, metadata}` 문서로 변환
4. 에러코드는 RDB(SQLite)에, 매뉴얼 청크는 벡터DB(Chroma)에 저장하는 하이브리드 구조
5. 질문에 에러코드가 언급되면 RDB로, 증상만 설명하면 벡터 검색으로 라우팅하는 질의응답

대상 문서: LG 벽걸이 에어컨 `FQ18GU1BHN` 사용설명서 (64페이지, LG 공식 고객지원
페이지에서 다운로드) + LG 공식 에어컨 에러코드 안내 페이지(11개 에러코드).


## 1. OCR이 필요한 문서인가? — 진단 결과

In [1]:
from pipeline.pdf_chunker import ocr_recommended

diag = ocr_recommended("data/manuals_pdf/AC_FQ18GU1BHN.pdf")
diag

{'avg_text_chars_per_page': 596.6,
 'avg_images_per_page': 0.0,
 'ocr_recommended': False,
 'reason': '텍스트 레이어가 정상적으로 존재함 -> OCR 불필요'}

**결론: 이 문서는 OCR이 필요 없다.**

- 페이지당 평균 596.6자의 텍스트가 정상적으로 추출됨 (스캔본이면 거의 0에 가까움)
- 페이지당 이미지가 평균 0개 — 리모컨/제품 다이어그램도 래스터 이미지가 아니라
  벡터 그래픽 + 실제 텍스트 라벨로 구성되어 있어서, `pymupdf`의 텍스트 추출만으로
  다이어그램 라벨까지 전부 잡힘 (64페이지 전수 확인, 이미지가 있는 페이지는 단 2곳뿐이고
  그마저도 QR코드/아이콘 수준)
- **`ocr_recommended()` 함수는 범용으로 만들어서, 스캔본 매뉴얼(오래된 제품, 사진으로
  찍은 문서 등)을 다룰 때는 자동으로 OCR 필요 여부를 진단할 수 있음** — 이번 문서는
  해당 없었을 뿐, 판단 로직 자체는 재사용 가능


## 2. 청킹 방식 비교: 폰트 크기 추정 vs PDF 내장 목차(TOC)

In [1]:
from pipeline.pdf_chunker import extract_sections_by_font, extract_sections_from_toc

font_sections = extract_sections_by_font("data/manuals_pdf/AC_FQ18GU1BHN.pdf")
print(f"폰트 기반: {len(font_sections)}개 섹션")
for s in font_sections[3:9]:
    print(f"  p{s['page']:>2} [{s['heading']}] ({len(s['text'])}자)")

폰트 기반: 54개 섹션
  p 4 [안전을 위해 주의하기] (228자)
  p 4 [LG ThinQ 사용하기] (81자)
  p 4 [알아보기] (60자)
  p 4 [리모컨으로 사용하기] (306자)
  p 4 [조작부로 사용하기] (77자)
  p 4 [관리하기] (149자)


폰트 기반 방식의 문제: 목차 페이지(2~3페이지)의 항목들이 실제 제목처럼 오검출되고,
안전 경고문 구간(4~10페이지)은 "R 금지 사항"/"j 준수 사항" 같은 **반복되는 하위
라벨까지 전부 새 섹션 경계로 잘못 인식**해서 의미 없이 잘게 쪼개진다.

In [1]:
toc_sections = extract_sections_from_toc("data/manuals_pdf/AC_FQ18GU1BHN.pdf")
print(f"TOC 기반: {len(toc_sections)}개 섹션\n")
for s in toc_sections:
    print(f"  p{s['page']:>2} [{s['heading']}] ({len(s['text'])}자)")

TOC 기반: 19개 섹션

  p 1 [제품 사용설명서] (1494자)
  p 4 [경고] (4163자)
  p 9 [주의] (943자)
  p11 [LG ThinQ와 LG 가전 연결하기] (2190자)
  p13 [에어컨의 모습과 기능 살펴보기] (2126자)
  p17 [리모컨 살펴보기] (3388자)
  p22 [냉방 기본 기능 작동하기] (4825자)
  p28 [공기청정 기능 사용하기] (1932자)
  p31 [추가 기능 및 설정하기] (2299자)
  p35 [조작부 살펴보기] (1335자)
  p37 [추가 기능 및 설정하기] (742자)
  p38 [청소하기] (8511자)
  p51 [냄새 제거하기] (1238자)
  p52 [보관하기] (668자)
  p53 [고장 진단하기] (380자)
  p54 [문제 해결하기] (5177자)
  p58 [제품 보증서] (3506자)
  p61 [제품 규격] (841자)
  p62 [생활 속 전기안전 캠페인] (974자)


### 왜 TOC 방식이 더 나은가

PDF에는 내비게이션용 북마크(목차)가 실제로 내장되어 있었다 (`doc.get_toc()`로 확인,
150개 항목, 레벨 1~4의 계층 구조). 이 정보를 그대로 활용하면:

1. **`R 금지 사항` / `j 준수 사항` 같은 반복 라벨은 레벨 4(가장 깊은 계층)** 라는 게
   TOC에 명시돼 있어서, "레벨 2까지만 청크 경계로 삼고 그보다 깊은 레벨은 상위
   섹션 본문에 합친다"는 규칙을 적용할 수 있다 → "경고" 섹션 하나로 깔끔하게 통합됨
   (4163자, 이전엔 수십 개의 조각으로 쪼개졌었음)
2. 시행착오 하나: 처음엔 "반복 빈도가 3회 이상인 제목은 일반 라벨로 취급"하는
   방식으로 시도했는데, **같은 페이지에 여러 레벨의 제목이 몰려있는 경우
   (예: 레벨1 제목 바로 다음 줄에 레벨2 제목) 페이지 범위가 겹쳐서 중복 청크가
   129개나 생기는 부작용**이 있었다. 레벨 기준으로 바꾸고, 겹치는 텍스트가
   완전히 포함관계면 제거하는 후처리를 추가해서 해결함 (19개로 정리됨).
3. 실제 목차 구조와 최종 청크가 정확히 일치: `LG ThinQ 연결하기`, `냉방 기본 기능`,
   `청소하기`, `문제 해결하기` 등 — 사람이 매뉴얼을 넘겨볼 때 보는 단위 그대로.

**일반화**: TOC가 없는 PDF(스캔본이거나 북마크를 안 넣은 문서)는 자동으로 폰트 기반
방식으로 폴백한다 (`extract_sections()`가 이 분기를 알아서 처리).

## 3. 청크를 문서({id, text, metadata})로 변환

In [1]:
from pipeline.pdf_chunker import build_documents_from_pdf

docs = build_documents_from_pdf(
    "data/manuals_pdf/AC_FQ18GU1BHN.pdf",
    product_model="FQ18GU1BHN",
    product_name="LG 벽걸이 에어컨 FQ18GU1BHN",
    category="에어컨",
)
print(f"최종 문서(청크) 수: {len(docs)}개  (19개 섹션을 700자 단위로 추가 분할)")
docs[10]

최종 문서(청크) 수: 77개  (19개 섹션을 700자 단위로 추가 분할)


{'id': 'FQ18GU1BHN_manual_5_0',
 'text': '[LG 벽걸이 에어컨 FQ18GU1BHN / 사용설명서 / 리모컨 살펴보기] ...(생략)...',
 'metadata': {'product_model': 'FQ18GU1BHN',
  'product_name': 'LG 벽걸이 에어컨 FQ18GU1BHN',
  'category': '에어컨',
  'section_type': '사용설명서',
  'section_title': '리모컨 살펴보기',
  'page': 17}}

## 4. 하이브리드 저장 구조: 에러코드 → RDB, 매뉴얼(+에러코드 텍스트) → 벡터DB

**왜 둘 다 필요한가**: 에러코드는 "코드를 입력하면 정해진 조치법 하나"라는 명확한
key-value 조회라서 RDB가 맞다. 반면 "에어컨에서 냄새 나요" 같은 증상 설명은 조회할
'키'가 없으니(코드가 없으니) 의미 기반 검색(벡터DB)이 필요하다.

실제로 로컬 3B 모델로 36개 질문을 평가하다가 두 가지 실패를 발견했다:
- `"CH04랑 FL이랑 같은 에러인가요?"` → 벡터 검색 결과를 LLM이 잘못 해석해서
  **"다른 코드"라고 오답** (원문엔 같은 코드의 다른 표시일 뿐이라고 나와있음)
- `"P6 에러가 뭔지 알려줘요"` → 벡터 검색이 긴 청크 속에 묻힌 별칭까지는
  못 찾아서 **"찾을 수 없음"으로 검색 자체가 누락**

이 두 문제를 RDB 정확 조회(코드 별칭 테이블)로 해결했다.

In [1]:
from db.error_codes_db import lookup_code

lookup_code("P6")

[{'solution_id': 6,
 'category': '에어컨',
 'title': 'CH61 (실외기/실내기 온도 이상, 제품에 따라 P4/P6/P7/P8/CH34)',
 'content': 'CH61 에러는 냉방 운전 중 실외기 온도가 높거나...(생략)',
 'all_aliases': 'CH34,CH61,P4,P6,P7,P8'}]

## 5. 질의응답 라우터 동작 확인 — 실제 OpenAI(gpt-4o-mini) 결과

In [1]:
from db.error_codes_db import find_codes_in_text

for q in ["CH04랑 FL이랑 같은 에러인가요?", "P6 에러가 뭔지 알려줘요", "에어컨에서 냄새 나요"]:
    print(f"{q!r} -> 감지된 코드: {find_codes_in_text(q)}")

'CH04랑 FL이랑 같은 에러인가요?' -> 감지된 코드: ['CH04', 'FL']
'P6 에러가 뭔지 알려줘요' -> 감지된 코드: ['P6']
'에어컨에서 냄새 나요' -> 감지된 코드: []


In [1]:
from pipeline.query_engine import answer

r = answer("CH04랑 FL이랑 같은 에러인가요?")
print(f"[경로: {r['source']}]")
print(r["answer"])

[경로: rdb]
네, CH04와 FL은 같은 에러 코드로, 둘 다 배수 불량 및 만수 감지와 관련된 경고입니다. 이 에러는 조건에 따라 다르게 나타날 수 있지만, 기본적으로 실내기에서 생성된 물이 제대로 배출되지 않을 때 발생합니다. 천장형 에어컨에서는 배수 문제가 발생할 수 있고, 창호형이나 이동식 에어컨에서는 주로 습도가 높거나 비가 오는 날에 일시적으로 나타날 수 있습니다. 그런 경우, 제품은 정상적으로 작동하므로 크게 걱정하지 않으셔도 됩니다. 하지만 맑은 날에도 자주 발생한다면 제품 점검이 필요합니다.


이전(로컬 3B, 벡터 검색만) 응답은 "CH04와 FL은 다른 에러코드"라고 잘못 답했었다. RDB 경로 + OpenAI로 바꾸니 정답이 나온다.

In [1]:
r = answer("P6 에러가 뭔지 알려줘요")
print(f"[경로: {r['source']}]")
print(r["answer"])

[경로: rdb]
P6 에러는 CH61 코드와 관련이 있으며, 주로 냉방 운전 중 실외기 온도가 높거나 난방 운전 중 실내기 온도가 높을 때 발생합니다.

### 원인:
1. **냉방 운전 중**: 실외기가 설치된 장소에 환기가 제대로 이루어지고 있지 않아, 실외기실의 온도가 높아져 에러가 발생합니다.
2. **난방 운전 중**: 실내기 내부 온도가 높아지는 경우, 보통 필터 청소가 제대로 이루어지지 않은 경우 나타납니다.

### 조치 방법:
1. 환기창이나 방충망이 닫혀 있다면 열어주세요.
2. 실외기 주변 장애물을 제거해 환기가 잘 되도록 하세요.
3. 실외기가 환기창보다 낮게 설치되어 있다면 받침대로 높이를 맞춰주세요.
4. 실내기 필터에 먼지가 많다면 청소해주세요.

P6 에러는 자동으로 해제되지 않으므로, 전원 플러그를 뽑았다가(또는 차단기를 내렸다가) 3분 이후에 재연결/재가동 해야 합니다.


이전엔 "찾을 수 없음"으로 검색 자체가 누락됐던 케이스인데, RDB 별칭 조회로 정확히 찾아낸다.

In [1]:
r = answer("에어컨에서 냄새 나요")
print(f"[경로: {r['source']}]")
print(r["answer"])

[경로: vector]
에어컨에서 냄새가 날 경우, 냄새의 종류에 따라 다음과 같은 조치를 취할 수 있습니다.

1. 매캐하고 쾨쾨한 냄새: 창문을 열고 환기하면서 공기청정/송풍 기능으로 1시간 이상 운전하세요. AI 건조 기능을 설정하면 냄새를 줄일 수 있습니다.
2. 플라스틱, 금속 냄새: 창문을 열고 환기하면서 1시간 이상 냉방 운전을 하세요.
3. 시큼함/매움/석유/새집 냄새: 환기 후 AI 건조나 공기청정 운전, 필터 청소(계속되면 교체), 열교환기 세척 서비스를 이용하세요.
4. 음식 냄새: 강한 냄새가 나는 요리 중엔 사용을 피하고, 요리 후 환기한 다음 사용하세요.
5. 화장실/하수구 냄새: 배수 호스 끝부분 위치를 확인하고, 환기 및 필터 청소를 하세요.

문제가 지속되면 전원을 끄고 제조사 서비스센터에 문의하시기 바랍니다.


이건 애초에 에러코드가 없는 증상 질문이라 라우터가 자동으로 벡터 검색으로
넘어갔고, 매뉴얼의 "냄새 제거하기" 섹션(냄새 종류별 세부 분류까지) 내용을 정확히
찾아서 답했다.

## 6. 정리

| 항목 | 결론 |
|---|---|
| OCR 필요 여부 | 이 문서는 불필요 (텍스트 레이어 정상). `ocr_recommended()`로 향후 다른 문서는 자동 진단 가능 |
| 청킹 방식 | PDF 내장 TOC 기반이 폰트 추정보다 훨씬 정확함 (54개의 잘게 쪼개진 조각 → 19개의 의미 단위) |
| 저장 구조 | 에러코드=RDB(정확 조회), 매뉴얼+증상형 질문=벡터DB — 실제 실패 사례(CH04/FL 오답, P6 검색누락) 2건을 근거로 결정 |
| LLM 백엔드 | 로컬 3B는 RDB로 정확한 원문을 줘도 요약 과정에서 사실을 틀리는 경우가 있었음(P6 온도 방향 반전 등) → 안전이 걸린 조치법 도메인이라 OpenAI(gpt-4o-mini)로 전환 |

**다음 단계 후보**: 여러 PDF(냉장고/세탁기 등) 배치 처리, 삼성 매뉴얼까지 확장,
`문제 해결하기` 섹션처럼 긴 청크(5000자+) 내부를 더 세분화할지 검토.